<a href="https://colab.research.google.com/github/carolineb3/Earnings-Call-NLP-Pipeline/blob/main/notebooks/02_spark_infrastructure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 2: Spark Join and Return Labeling

This notebook takes the transcript JSON files and stock return data from Step 1 and brings them into Spark. The goal is to join each transcript to its earnings-date return data, create the positive/negative return label, and save the first clean Parquet checkpoint for the rest of the pipeline.

The label is based on the 3-day market-adjusted return:

- `positive` if the return is greater than +0.5%
- `negative` if the return is less than -0.5%
- `neutral` if the return is between -0.5% and +0.5%

Neutral rows are removed because the model is focused on clearer directional reactions rather than near-zero market moves.

## Inputs

- `Transcripts_JSON/*.json`
- `stock_prices.csv`

## Output

- `584_earnings_final.parquet`

## Checks in this notebook

- Confirms the transcript and stock price data loaded correctly.
- Prints the return-label distribution before neutral rows are removed.
- Prints the final labeled observation count.
- Saves and reloads the Parquet file to confirm the checkpoint was written successfully.

In [ ]:
# =============================================================================
# STEP 2 — Spark Join and Return Labeling
# Join transcript JSONs with stock return data, create the binary return label,
# and save the first Parquet checkpoint for the modeling pipeline.
# =============================================================================

!pip install pyspark

from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local") \
    .appName("584_Spark_Infrastructure") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark version:", spark.version)

import os

BASE_DIR        = "/content/drive/MyDrive/MIS_584_Project"
TRANSCRIPTS_DIR = f"{BASE_DIR}/Transcripts_JSON"
PRICE_CSV       = f"{BASE_DIR}/stock_prices.csv"
PARQUET_DIR     = f"{BASE_DIR}/Parquet"

os.makedirs(PARQUET_DIR, exist_ok=True)

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

price_schema = StructType([
    StructField('ticker',               StringType(),  True),
    StructField('earnings_date',        StringType(),  True),
    StructField('price_on_date',        DoubleType(),  True),
    StructField('price_3d_later',       DoubleType(),  True),
    StructField('pct_return_3d',        DoubleType(),  True),
    StructField('market_adj_return_3d', DoubleType(),  True),
    StructField('spy_return_3d',        DoubleType(),  True),
    StructField('volume_on_date',       IntegerType(), True),
    StructField('window_days_used',     IntegerType(), True),
])

prices_df = spark.read.csv(PRICE_CSV, header=True, schema=price_schema)
print("Stock Prices Schema:")
prices_df.printSchema()
prices_df.show()

# Read all JSON files into one Spark DataFrame
# multiLine=True because each file is one JSON object
transcripts_df = (
    spark.read
    .option("multiLine", "true")
    .json(f"{TRANSCRIPTS_DIR}/*.json")
    .drop("structured_content")
)

print(f"Total transcripts loaded: {transcripts_df.count()}")
transcripts_df.select('ticker', 'quarter', 'earnings_date', 'word_count').show(10)

# ── Remove duplicates and short/empty transcripts ───────────────────────
transcripts_df = transcripts_df.dropDuplicates()
prices_df      = prices_df.dropDuplicates()

transcripts_df = transcripts_df.filter(
    transcripts_df['full_text'].isNotNull() &
    (transcripts_df['word_count'] > 100)
)

# ── Join and label ────────────────────────────────────────────────────────────
from pyspark.sql.functions import when, col, avg, round as spark_round, stddev

joined_raw = transcripts_df.join(
    prices_df, on=['ticker', 'earnings_date'], how='inner'
).withColumn(
    'return_label',
    when(col('market_adj_return_3d') >  0.5, 'positive')
    .when(col('market_adj_return_3d') < -0.5, 'negative')
    .otherwise('neutral')
)

# ── Check label counts before removing neutral rows ──────────────────────────
total_joined = joined_raw.count()
pos_count    = joined_raw.filter(col('return_label') == 'positive').count()
neg_count    = joined_raw.filter(col('return_label') == 'negative').count()
neu_count    = joined_raw.filter(col('return_label') == 'neutral').count()
pct_removed  = neu_count / total_joined * 100

print(f"\n{'='*55}")
print(f"  Return Label Distribution (before neutral filter)")
print(f"{'='*55}")
print(f"  Total joined observations    : {total_joined:,}")
print(f"  Positive  (> +0.5% adj.)     : {pos_count:,}  ({pos_count/total_joined*100:.1f}%)")
print(f"  Negative  (< -0.5% adj.)     : {neg_count:,}  ({neg_count/total_joined*100:.1f}%)")
print(f"  Neutral   (within ±0.5%)      : {neu_count:,}  ({pct_removed:.1f}% removed)")

joined_df = joined_raw.filter(col('return_label') != 'neutral')
print(f"\n  Final labeled observations   : {joined_df.count():,}")
print(f"{'='*55}")
joined_df.select(
    'ticker', 'quarter', 'earnings_date',
    'pct_return_3d', 'market_adj_return_3d', 'return_label'
).show()

# ── Basic Spark checks before saving ─────────────────────────────────────────
joined_df.createOrReplaceTempView("584_earnings")

print("Average word count and return by ticker:")
joined_df.groupby('ticker').agg(
    spark_round(avg('word_count'),          0).alias('avg_word_count'),
    spark_round(avg('pct_return_3d'),       4).alias('avg_return_pct'),
    spark_round(avg('market_adj_return_3d'),4).alias('avg_adj_return'),
    spark_round(stddev('pct_return_3d'),    4).alias('std_return_pct')
).sort('ticker').show()

print("Top market-adjusted positive returns:")
spark.sql("""
    SELECT ticker, quarter, earnings_date,
        pct_return_3d, market_adj_return_3d, return_label
    FROM 584_earnings
    WHERE market_adj_return_3d > 0.5
    ORDER BY market_adj_return_3d DESC
""").show()

parquet_path = f"{PARQUET_DIR}/584_earnings_final"
joined_df.write.mode('overwrite').parquet(parquet_path)
print(f"Saved to: {parquet_path}")
print("Verification rows:", spark.read.parquet(parquet_path).count())

spark.stop()
print("Spark session stopped.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark version: 4.0.2
Stock Prices Schema:
root
 |-- ticker: string (nullable = true)
 |-- earnings_date: string (nullable = true)
 |-- price_on_date: double (nullable = true)
 |-- price_3d_later: double (nullable = true)
 |-- pct_return_3d: double (nullable = true)
 |-- market_adj_return_3d: double (nullable = true)
 |-- spy_return_3d: double (nullable = true)
 |-- volume_on_date: integer (nullable = true)
 |-- window_days_used: integer (nullable = true)

+------+-------------+-------------+--------------+-------------+--------------------+-------------+--------------+----------------+
|ticker|earnings_date|price_on_date|price_3d_later|pct_return_3d|market_adj_return_3d|spy_return_3d|volume_on_date|window_days_used|
+------+-------------+-------------+--------------+-------------+--------------------+-------------+--------------+----------------+
|     A|   2